# Día 6 · Unidad 6 — Código profesional (V): Git y GitHub

**Objetivos de aprendizaje**
- Explicar qué es un sistema de control de versiones distribuido y qué problema resuelve.
- Usar el flujo básico local: `init`, `add`, `commit`, `status`, `log`, y entender el área de *staging*.
- Usar `.gitignore` para excluir archivos que no deben versionarse (secretos, artefactos generados).
- Moverte en el historial (`checkout`, `HEAD`, SHA) y deshacer cambios de forma segura con `revert`.
- Trabajar con ramas: crear, `merge`, `rebase`, `cherry-pick`.
- Sincronizar con un repositorio remoto: `clone`, `fetch`, `pull`, `push`, y entender qué es un Pull Request.
- Aplicar un checklist propio antes de hacer commit (bloque de IA de este notebook).

**Duración estimada:** 120-150 minutos — es el bloque más largo del día.

**Requisitos previos:** ninguno de Python en concreto. Necesitas tener `git` instalado (comprobado en el Día 1, `docs/guia_alumno_setup.md`).

## 1. ¿Qué es Git?

Git es un **sistema de control de versiones distribuido (DVCS)**:

- **Sistema de control de versiones**: un conjunto de herramientas que registra el historial de cambios de un conjunto de archivos (un *repositorio*). Permite comparar versiones y recuperar cualquier archivo (o todos) de cualquier punto del historial.
- **Distribuido**: no existe un servidor central con "la versión buena". Cada persona que clona un repositorio tiene una copia completa del historial, no solo del último estado.

> 📊 En Excel esto sería como tener, en vez de un único archivo que vas sobrescribiendo (y del que quizá guardas copias sueltas `informe_v2_final_definitivo.xlsx`), un historial completo y ordenado de cada versión guardada, con quién la guardó, cuándo, y por qué — y la posibilidad de volver a cualquiera de ellas con un comando, no buscando en una carpeta de copias.

## 2. Preparar una demo reproducible

Todo lo que sigue en este notebook ocurre dentro de **carpetas temporales**, creadas con el módulo estándar `tempfile` — nunca dentro del repositorio del curso. Así, este notebook se puede ejecutar tantas veces como haga falta (es *idempotente*): cada ejecución crea sus propias carpetas nuevas y las borra al final, sin dejar rastro ni depender de un estado previo.

Definimos también una pequeña función `run()` que ejecuta un comando y muestra su salida, como si lo escribieras en una terminal:

In [ ]:
import os
import shutil
import stat
import subprocess
import tempfile
from pathlib import Path


def run(cmd: list[str], cwd: str | None = None) -> subprocess.CompletedProcess:
    """Ejecuta un comando de shell y muestra su salida, como en una terminal."""
    resultado = subprocess.run(
        cmd, cwd=cwd, capture_output=True, text=True, encoding="utf-8", errors="replace"
    )
    print(f"$ {' '.join(cmd)}")
    salida = (resultado.stdout or "") + (resultado.stderr or "")
    print(salida.rstrip())
    print()
    return resultado


def eliminar_forzado(func, path, exc_info):
    """Handler para shutil.rmtree: en Windows, algunos archivos internos de .git
    quedan de solo lectura y el borrado normal falla. Quitamos esa marca y reintentamos."""
    os.chmod(path, stat.S_IWRITE)
    func(path)


demo_dir = tempfile.mkdtemp(prefix="curso_python_git_demo_")
print("Repositorio de demostración (desechable):", demo_dir)

## 3. Crear un repositorio: `init`, `config`, `status`

`git init` convierte una carpeta normal en un repositorio Git (crea una subcarpeta oculta `.git/` con todo el historial). Usamos `-b main` para fijar el nombre de la rama inicial explícitamente, sin depender de la configuración global de quien ejecute el notebook.

In [ ]:
run(["git", "init", "-b", "main"], cwd=demo_dir)

# Configuramos nombre y email SOLO para este repositorio (sin tocar tu configuracion global de git)
run(["git", "config", "user.name", "Alumna CAC"], cwd=demo_dir)
run(["git", "config", "user.email", "alumna@example.com"], cwd=demo_dir)

run(["git", "status"], cwd=demo_dir);

`git status` muestra: en qué rama estamos (`main`), que no hay commits todavía, y ningún archivo (la carpeta está vacía). Vamos a crear un archivo real.

## 4. Añadir archivos y hacer el primer commit

### Staging: `add`

El área de ***staging*** (o *index*) es donde Git guarda qué cambios quieres que formen parte del **próximo** commit. `git add` mueve un archivo del estado "sin seguimiento" al *staging*.

In [ ]:
(Path(demo_dir) / "tarificacion.py").write_text("PRIMA_BASE = 300.0\n", encoding="utf-8")

run(["git", "status"], cwd=demo_dir)  # tarificacion.py aparece como "Untracked files"

run(["git", "add", "tarificacion.py"], cwd=demo_dir)

run(["git", "status"], cwd=demo_dir);  # ahora aparece en "Changes to be committed" 

### Commit

Un *commit* es una "fotografía" del estado actual del área de staging, con un mensaje que explica el cambio. Cada commit queda identificado por un **SHA** (sección 6).

In [ ]:
run(["git", "commit", "-m", "feat: add base premium constant"], cwd=demo_dir)

commit_1 = run(["git", "rev-parse", "HEAD"], cwd=demo_dir).stdout.strip()
run(["git", "log", "--oneline"], cwd=demo_dir);

## 5. Las tres versiones de un archivo

En un momento dado, un archivo puede existir en tres estados distintos a la vez:

1. **La versión en tu disco** (lo que ves si abres el archivo ahora).
2. **La versión en *staging*** (lo último que añadiste con `git add`).
3. **La versión del último commit** (la última "fotografía" confirmada).

Vamos a verlo en acción: modificamos el archivo, lo añadimos a staging, y lo modificamos otra vez **sin** volver a añadirlo:

In [ ]:
(Path(demo_dir) / "tarificacion.py").write_text(
    "PRIMA_BASE = 300.0\nRECARGO_GESTION = 0.15\n", encoding="utf-8"
)
run(["git", "status"], cwd=demo_dir)  # modificado, todavia NO en staging

run(["git", "add", "tarificacion.py"], cwd=demo_dir)
run(["git", "status"], cwd=demo_dir)  # ahora SI en staging ("Changes to be committed")

(Path(demo_dir) / "tarificacion.py").write_text(
    "PRIMA_BASE = 300.0\nRECARGO_GESTION = 0.18\n", encoding="utf-8"
)
run(["git", "status"], cwd=demo_dir);  # aparece en AMBOS sitios a la vez: staged Y sin stagear

`git status` muestra el archivo dos veces: en "Changes to be committed" (la versión con `0.15`, la que quedó en *staging*) y en "Changes not staged for commit" (la versión con `0.18`, la que hay ahora en disco). Si haces commit ahora, se guardará la versión de *staging* (`0.15`), no la más reciente del disco. Añadimos la última versión y confirmamos:

In [ ]:
run(["git", "add", "tarificacion.py"], cwd=demo_dir)
run(["git", "commit", "-m", "feat: increase management surcharge"], cwd=demo_dir)

commit_2 = run(["git", "rev-parse", "HEAD"], cwd=demo_dir).stdout.strip()
run(["git", "log", "--oneline"], cwd=demo_dir);

## 6. `.gitignore`: qué no versionar

A veces hay archivos que **no** deben entrar en el repositorio — típicamente, credenciales. En la práctica esto se gestiona con variables de entorno (Unidad 9), no con archivos `.py` con secretos dentro, pero simulemos el caso para ver cómo actúa `.gitignore`:

In [ ]:
(Path(demo_dir) / "credenciales.py").write_text("API_KEY = 'no-es-un-secreto-real'\n", encoding="utf-8")
run(["git", "status"], cwd=demo_dir);  # credenciales.py aparece como untracked

In [ ]:
(Path(demo_dir) / ".gitignore").write_text("credenciales.py\n", encoding="utf-8")
run(["git", "status"], cwd=demo_dir)  # credenciales.py ya NO aparece; .gitignore si, como nuevo archivo

run(["git", "add", ".gitignore"], cwd=demo_dir)
run(["git", "commit", "-m", "chore: ignore local credentials file"], cwd=demo_dir)

commit_3 = run(["git", "rev-parse", "HEAD"], cwd=demo_dir).stdout.strip()

> ⚠️ **Error típico**: añadir `.gitignore` **después** de haber hecho ya `git add` y `commit` de un archivo con secretos. `.gitignore` solo afecta a archivos que Git todavía no sigue — si un archivo con credenciales ya fue commiteado, seguirá en el historial (recuperable con `git show <commit>:<archivo>`) aunque lo borres o lo ignores después. Si eso pasa de verdad, hay que rotar la credencial expuesta, no solo borrar el archivo.

## ¿Qué no añadir a un repositorio?

Regla general: añade solo **source files** (lo que escribes tú), nunca **archivos generados** (lo que crea el ordenador a partir de esos archivos: `__pycache__/`, un `.db` generado, un notebook con outputs pesados). Y nunca información confidencial — pólizas reales, credenciales, datos de clientes — ni siquiera en un repositorio privado.

## 7. Bloque de IA: checklist antes de hacer commit

Antes de confirmar un commit real (en cualquier proyecto, no solo en este curso), es buena práctica revisar el `diff` con una lista de comprobación. Un asistente de IA puede ayudarte a repasarla — pero, como siempre en este curso, la responsabilidad de decidir si el commit está listo sigue siendo tuya, no de la IA.

In [ ]:
prompt_checklist_commit = """
Contexto: voy a hacer commit de estos cambios en un proyecto Python de un curso para actuarios.
Aqui tienes la salida de `git diff --staged` de mi repositorio:

<pega aqui la salida real de `git diff --staged` de tu propio repositorio>

Objetivo: antes de que confirme el commit, revisa el diff y responde a esto:
1. Hay prints de depuracion olvidados (print(), breakpoint(), pdb.set_trace(), console.log)?
2. Hay algo que parezca un dato sensible: contraseñas, tokens, rutas personales,
   datos de clientes o polizas?
3. El mensaje de commit que te propongo, mas abajo, explica el PORQUE del cambio
   y no solo el QUE?

Mensaje de commit propuesto: "<pega aqui tu mensaje de commit>"

Restricciones: no reescribas mi codigo, señala solo los problemas concretos
(con numero de linea si puedes).
Formato de salida: lista numerada de hallazgos, o "Sin hallazgos" si no ves ninguno.
"""
print(prompt_checklist_commit)

**Qué esperar**: una lista corta señalando, si los hay, `print()` sueltos, algo que parezca una credencial o dato personal, y una valoración de si el mensaje de commit explica el motivo del cambio (no solo repite qué archivos ha tocado).

**Qué verificar**: que revises tú mismo el diff aunque la IA responda "Sin hallazgos" — un asistente no conoce el contexto de negocio completo (por ejemplo, no sabe si un número de póliza en un archivo de prueba es realmente sintético o se coló uno real de un caso de trabajo), y las alucinaciones también aplican aquí: puede pasar por alto algo real o señalar como "sensible" algo que no lo es. El checklist acelera la revisión, no la sustituye — el mismo principio de `docs/uso_ia_en_el_curso.md` que ya conoces.

Como referencia rápida, el checklist mínimo antes de cualquier commit, con o sin ayuda de IA:

- [ ] ¿Hay `print()`/`breakpoint()` de depuración que se me han olvidado quitar?
- [ ] ¿El mensaje de commit explica el **porqué**, no solo el qué (`git commit -m "arreglo bug"` no dice nada útil dentro de seis meses)?
- [ ] ¿Se ha colado algún dato sensible (credenciales, rutas personales, datos reales)?
- [ ] ¿Este commit hace **una sola cosa** coherente, o mezcla varios cambios sin relación?

## 8. SHA y `git log`

Cada commit tiene un identificador único (**SHA**, 40 caracteres hexadecimales) calculado a partir de su contenido, autor, fecha y del SHA del commit anterior — encadenando así todo el historial. `git log` muestra el historial completo:

In [ ]:
run(["git", "log"], cwd=demo_dir);

- `commit <SHA>`: el identificador completo de esa "fotografía".
- `Author` / `Date`: quién y cuándo.
- El mensaje del commit, debajo.

`git log --oneline` (usado en las secciones anteriores) es la versión compacta: un SHA corto y el mensaje, una línea por commit.

## 9. Viajar en el historial: `checkout` y `HEAD`

**`HEAD`** es el nombre que usa Git para el commit que estás viendo *ahora mismo*. Casi siempre `HEAD` coincide con la punta de una rama (`main`), pero se puede mover a cualquier commit del historial con `git checkout <SHA>`:

In [ ]:
run(["git", "checkout", commit_1], cwd=demo_dir)  # el primer commit, antes del recargo y el gitignore
run(["git", "status"], cwd=demo_dir)

print("Contenido de tarificacion.py en ese punto del historial:")
print((Path(demo_dir) / "tarificacion.py").read_text(encoding="utf-8"))

`git status` avisa de un **"detached HEAD"**: `HEAD` apunta a un SHA concreto que no tiene ninguna rama asociada — estás viendo el pasado, pero no "sobre" ninguna rama. Es perfectamente seguro para mirar (o incluso probar cosas sueltas), siempre que sepas cómo volver:

In [ ]:
run(["git", "checkout", "main"], cwd=demo_dir)
run(["git", "status"], cwd=demo_dir)

print("Contenido de tarificacion.py de vuelta en main:")
print((Path(demo_dir) / "tarificacion.py").read_text(encoding="utf-8"))

## 10. Deshacer un cambio de forma segura: `git revert`

`checkout` mueve `HEAD` a otro punto, pero no borra historial ni crea nada nuevo. Para **deshacer** un commit concreto sin reescribir el historial (fundamental si ese commit ya se compartió con otras personas), se usa `git revert`: crea un **commit nuevo** que aplica el cambio contrario.

Vamos a revertir `commit_2` (el que subió el recargo de gestión a `0.18`), aunque ya no sea el último commit — Git calcula el cambio inverso igualmente:

In [ ]:
run(["git", "revert", "--no-edit", commit_2], cwd=demo_dir)
run(["git", "log", "--oneline"], cwd=demo_dir)

print("Contenido de tarificacion.py tras el revert:")
print((Path(demo_dir) / "tarificacion.py").read_text(encoding="utf-8"))

El historial **no ha desaparecido** — el commit `commit_2` sigue ahí, visible en `git log`; simplemente hay un commit nuevo encima que deshace su efecto. Es la forma segura de "deshacer" en un repositorio compartido: `git reset` (que sí reescribe el historial moviendo la rama hacia atrás) puede ser peligroso si otras personas ya descargaron esos commits — `revert` nunca tiene ese problema.

**Para ti:** ejecuta `run(["git", "show", commit_2], cwd=demo_dir)` en una celda nueva y compara su salida con la del commit de revert que aparece al principio de `git log --oneline` de la celda anterior. ¿Qué relación ves entre ambos diffs?

In [ ]:
# Tu código aquí


## 11. Ramas (*branches*)

Las ramas permiten mantener líneas de desarrollo separadas — imprescindible al trabajar en equipo. Creamos una rama para desarrollar una funcionalidad sin tocar `main` directamente:

In [ ]:
run(["git", "checkout", "-b", "feature/recargo-autos"], cwd=demo_dir)
run(["git", "status"], cwd=demo_dir)  # ahora estamos en la nueva rama

(Path(demo_dir) / "recargo_autos.py").write_text("RECARGO_AUTO = 0.10\n", encoding="utf-8")
run(["git", "add", "recargo_autos.py"], cwd=demo_dir)
run(["git", "commit", "-m", "feat: add car insurance surcharge module"], cwd=demo_dir);

Mientras tanto, en `main` (simulando que otra persona del equipo sigue trabajando ahí en paralelo) añadimos un cambio distinto, para que las dos ramas **diverjan** de verdad:

In [ ]:
run(["git", "checkout", "main"], cwd=demo_dir)

(Path(demo_dir) / "provisiones.py").write_text("PROVISION_MINIMA = 1000.0\n", encoding="utf-8")
run(["git", "add", "provisiones.py"], cwd=demo_dir)
run(["git", "commit", "-m", "feat: add minimum provision constant"], cwd=demo_dir)

run(["git", "log", "--oneline", "--all", "--graph"], cwd=demo_dir);

El `--graph` deja ver las dos líneas: `main` con su propio commit nuevo, `feature/recargo-autos` con el suyo, ambas partiendo de un antepasado común.

### Merge

`git merge` integra los commits de una rama en la actual. Como `main` y `feature/recargo-autos` han divergido (cada una tiene commits que la otra no tiene), Git crea un **commit de merge** que combina ambas historias — forzamos `--no-ff` para que quede explícito incluso si hubiera sido posible un *fast-forward*:

In [ ]:
run(["git", "merge", "--no-ff", "feature/recargo-autos", "-m", "merge: integrate car surcharge feature"], cwd=demo_dir)
run(["git", "log", "--oneline", "--graph"], cwd=demo_dir)

print("Archivos en main tras el merge:", sorted(os.listdir(demo_dir)))

`main` ahora tiene los archivos de las dos líneas de trabajo. Si la misma sección del mismo archivo se hubiera modificado en ambas ramas, Git no sabría cuál de las dos versiones quedarse — eso es un ***merge conflict***, y hay que resolverlo a mano (editando el archivo y volviendo a hacer `add`+`commit`). No lo forzamos aquí para mantener el notebook determinista, pero es importante saber que existe.

### Rebase

`rebase` es una alternativa a `merge`: en vez de crear un commit que combina dos historias, Git **reescribe** los commits de una rama para que parezcan haberse creado a partir de la punta actual de la otra — el historial queda lineal, sin commit de merge.

In [ ]:
run(["git", "checkout", "-b", "feature/notas-provision"], cwd=demo_dir)
(Path(demo_dir) / "notas_provision.md").write_text("# Notas sobre la provision minima\n", encoding="utf-8")
run(["git", "add", "notas_provision.md"], cwd=demo_dir)
run(["git", "commit", "-m", "docs: add notes about minimum provision"], cwd=demo_dir)

# Mientras tanto, main sigue avanzando (otro commit nuevo)
run(["git", "checkout", "main"], cwd=demo_dir)
(Path(demo_dir) / "readme_interno.md").write_text("# Notas internas del equipo\n", encoding="utf-8")
run(["git", "add", "readme_interno.md"], cwd=demo_dir)
run(["git", "commit", "-m", "docs: add internal team notes"], cwd=demo_dir)

# Volvemos a la rama y la "rebasamos" sobre la punta actual de main
run(["git", "checkout", "feature/notas-provision"], cwd=demo_dir)
run(["git", "rebase", "main"], cwd=demo_dir)

run(["git", "log", "--oneline", "--graph", "--all"], cwd=demo_dir);

> ⚠️ **Error típico**: hacer `rebase` de una rama que **ya se ha compartido** con otras personas (por ejemplo, ya subida a un Pull Request abierto). Rebase reescribe los SHA de los commits — quien ya tenía la versión anterior descargada se encuentra con un historial "distinto" e incompatible. Regla práctica: rebase libremente en tu rama local antes de compartirla; una vez compartida, usa `merge`.

### Cherry-pick

`cherry-pick` copia un commit **concreto** (por su SHA) de una rama a otra, sin traer el resto de la rama. Útil cuando solo necesitas un cambio puntual de otra línea de trabajo — por ejemplo, un fix urgente.

In [ ]:
run(["git", "checkout", "-b", "hotfix/limite-prima", "main"], cwd=demo_dir)
(Path(demo_dir) / "tarificacion.py").write_text(
    "PRIMA_BASE = 300.0\nRECARGO_GESTION = 0.15\nPRIMA_MAXIMA = 5000.0\n", encoding="utf-8"
)
run(["git", "add", "tarificacion.py"], cwd=demo_dir)
run(["git", "commit", "-m", "fix: cap maximum premium at safety threshold"], cwd=demo_dir)

sha_hotfix = run(["git", "rev-parse", "HEAD"], cwd=demo_dir).stdout.strip()

run(["git", "checkout", "main"], cwd=demo_dir)
run(["git", "cherry-pick", sha_hotfix], cwd=demo_dir)

print("Contenido de tarificacion.py en main tras el cherry-pick:")
print((Path(demo_dir) / "tarificacion.py").read_text(encoding="utf-8"))

`main` tiene ahora el límite de prima máxima, sin haber traído el resto de commits de `hotfix/limite-prima` (que en este caso solo tenía uno, pero podría haber tenido varios más sin relación con el fix).

**Para ti:** ¿en qué se diferencia, en la práctica, hacer `cherry-pick` de un commit frente a hacer `merge` de la rama entera que lo contiene? Piensa en un caso real: un compañero tiene una rama con 5 commits de una funcionalidad grande a medio terminar, pero uno de esos commits corrige un bug urgente que necesitas ya en `main`. Escribe tu respuesta en la celda de markdown siguiente.

_Tu respuesta aquí._

## 12. Repositorios remotos

Todo lo anterior ha sido local. Un repositorio **remoto** es una copia del repositorio alojada en otro sitio (un servidor propio, o un servicio como GitHub) con la que sincronizas cambios. Para no depender de una cuenta de GitHub real ni de conexión a internet, simulamos un remoto con un **repositorio bare** (`--bare`: solo contiene el historial, sin archivos de trabajo — así es exactamente como funciona un repositorio en GitHub) en otra carpeta temporal:

In [ ]:
remote_dir = tempfile.mkdtemp(prefix="curso_python_git_remote_")
run(["git", "init", "--bare", "-b", "main"], cwd=remote_dir)

run(["git", "remote", "add", "origin", remote_dir], cwd=demo_dir)
run(["git", "remote", "-v"], cwd=demo_dir);

### `push`

Envía los commits de tu rama al remoto y le pide que actualice su copia de esa rama:

In [ ]:
run(["git", "push", "-u", "origin", "main"], cwd=demo_dir)
run(["git", "push", "-u", "origin", "feature/recargo-autos"], cwd=demo_dir);

### `clone`

Crea una copia completa de un repositorio remoto (historial incluido) en una carpeta nueva — esto es lo que haría un compañero de equipo al incorporarse al proyecto por primera vez:

In [ ]:
clon_dir = str(Path(tempfile.mkdtemp(prefix="curso_python_git_clon_")) / "repositorio-clonado")
run(["git", "clone", remote_dir, clon_dir])

print("Archivos clonados:", sorted(os.listdir(clon_dir)))

### `fetch` y `pull`

`git fetch` descarga las novedades del remoto sin tocar tu rama local (actualiza solo las referencias `remotes/origin/...`). `git pull` es `fetch` + `merge` en un solo paso: descarga y además integra esos cambios en tu rama actual.

Simulamos que, tras el clon, alguien añade un commit nuevo en `demo_dir` y lo sube — y luego lo recuperamos desde el clon con `pull`:

In [ ]:
(Path(demo_dir) / "changelog.md").write_text("# Changelog\n- Added premium cap\n", encoding="utf-8")
run(["git", "add", "changelog.md"], cwd=demo_dir)
run(["git", "commit", "-m", "docs: add changelog"], cwd=demo_dir)
run(["git", "push", "origin", "main"], cwd=demo_dir)

print("Antes del pull:", sorted(os.listdir(clon_dir)))
run(["git", "pull", "origin", "main"], cwd=clon_dir)
print("Después del pull:", sorted(os.listdir(clon_dir)))

### Pull Requests

Un **Pull Request** (PR) es la forma habitual de integrar cambios en GitHub de manera estructurada: en vez de hacer `push` directo a `main`, se sube una rama (como ya hicimos con `feature/recargo-autos`) y se abre una solicitud para fusionarla, que permite:

- **Revisión**: compañeros comentan el código antes de integrarlo.
- **Workflows automáticos** (CI): comprobar que el formato (PEP 8), los tests (`pytest`) o el build pasan antes de poder fusionar.

No hay un remoto real de GitHub en este notebook (no depende de credenciales ni de red), pero el flujo con la rama `feature/recargo-autos` ya subida sería, en la práctica: abrir GitHub, ir al repositorio, y GitHub sugiere directamente "Compare & pull request" para esa rama — o, desde la terminal, con la [CLI de GitHub](https://cli.github.com/):

```shell
gh pr create --base main --head feature/recargo-autos \
  --title "feat: add car insurance surcharge" \
  --body "Añade el módulo de recargo para pólizas de auto. Incluye tests."
```

## 13. Limpieza

Borramos las tres carpetas temporales usadas en la demo. Recuerda: nada de esto ha tocado el repositorio real del curso — todo ha vivido en `tempfile.mkdtemp()`, fuera del árbol de trabajo de `curso-python-cac-main`.

In [ ]:
shutil.rmtree(demo_dir, onerror=eliminar_forzado)
shutil.rmtree(remote_dir, onerror=eliminar_forzado)
shutil.rmtree(os.path.dirname(clon_dir), onerror=eliminar_forzado)

print("demo_dir eliminado:", not os.path.exists(demo_dir))
print("remote_dir eliminado:", not os.path.exists(remote_dir))
print("clon_dir eliminado:", not os.path.exists(clon_dir))

**Para ti:** vuelve a ejecutar el notebook completo (`Kernel → Restart & Run All`). Comprueba que no da ningún error la segunda vez — eso es justo lo que significa que sea *idempotente*: no depende de haber quedado ningún estado de la ejecución anterior, porque todo vivía en carpetas temporales nuevas cada vez.

## Resumen

- Git es un sistema de control de versiones **distribuido**: cada copia clonada contiene el historial completo, no solo el último estado.
- El área de *staging* (`git add`) es un paso intermedio entre tu disco y el commit — en un momento dado, un archivo puede tener tres versiones distintas a la vez (disco / staging / último commit).
- `.gitignore` evita que Git siga archivos concretos (típicamente, secretos o artefactos generados) — pero no retira del historial nada que ya se haya commiteado antes.
- `HEAD` es el commit que estás viendo ahora; `checkout <SHA>` te mueve por el historial sin perder nada (aunque te deja en "detached HEAD").
- `git revert` deshace un commit creando uno nuevo que aplica el cambio contrario — a diferencia de `reset`, no reescribe el historial, así que es seguro en ramas ya compartidas.
- `merge`, `rebase` y `cherry-pick` son tres formas distintas de llevar commits de una rama a otra — cada una dejando el historial con una forma distinta (con commit de merge, lineal, o solo un commit suelto).
- Un repositorio remoto (`clone`/`fetch`/`pull`/`push`) sincroniza tu copia local con la de otra persona o servicio; un Pull Request añade revisión y comprobaciones automáticas antes de integrar una rama.
- Antes de cualquier commit, revisa un checklist mínimo (prints de depuración, mensaje que explique el porqué, ausencia de datos sensibles) — una IA puede ayudarte a repasarlo, pero la decisión final es tuya.

**Siguiente:** `06_ejercicios.ipynb`, con ejercicios que cubren todo el día: módulos, excepciones, logging y un ejercicio de escribir tus propios tests de pytest.